Step 1: Set Up Project 3 Environment & Create Dataset


In [1]:
import pandas as pd

# Define dataset of tech roles and associated skills/keywords
data = {
    'Job_Role': [
        'DevOps Engineer',
        'Cloud Architect',
        'Data Scientist',
        'Frontend Developer',
        'Backend Developer',
        'AI/ML Engineer',
        'Cybersecurity Specialist',
        'System Administrator'
    ],
    'Required_Skills': [
        'Python Cloud Automation Linux Docker Kubernetes CI/CD',
        'Cloud AWS Azure Infrastructure Automation Security Networks',
        'Python Machine Learning Data Analysis Statistics SQL Pandas',
        'JavaScript React HTML CSS Web Design UI/UX Frontend',
        'Python Java Node.js API Databases SQL Microservices',
        'Python Machine Learning Deep Learning PyTorch TensorFlow Algorithms',
        'Security Networks Linux Cryptography Ethical Hacking Python',
        'Linux Systems Administration Automation Bash Networks Hardware'
    ]
}

# Convert to DataFrame and save as CSV
df = pd.DataFrame(data)
df.to_csv('skills_dataset.csv', index=False)
print("Dataset created successfully!")
df.head()

Dataset created successfully!


,Job_Role,Required_Skills
0,DevOps Engineer,Python Cloud Automation Linux Docker Kubernete...
1,Cloud Architect,Cloud AWS Azure Infrastructure Automation Secu...
2,Data Scientist,Python Machine Learning Data Analysis Statisti...
3,Frontend Developer,JavaScript React HTML CSS Web Design UI/UX Fro...
4,Backend Developer,Python Java Node.js API Databases SQL Microser...


Step 2: Build the Recommendation Engine Logic (TF-IDF & Cosine Similarity)


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def recommend_job_roles(user_skills_list):
    # Load dataset
    df = pd.read_csv('skills_dataset.csv')

    # Combine user input into a single text query string
    user_query = " ".join(user_skills_list)

    # Combine job role skills with user input for unified vector mapping
    all_documents = list(df['Required_Skills']) + [user_query]

    # Initialize TF-IDF Vectorizer
    tfidf = TfidfVectorizer()
    tfidf_matrix = tfidf.fit_transform(all_documents)

    # Separate job vectors and the user profile vector
    job_vectors = tfidf_matrix[:-1]
    user_vector = tfidf_matrix[-1]

    # Compute Cosine Similarity between user profile and all job roles
    similarity_scores = cosine_similarity(user_vector, job_vectors).flatten()

    # Add similarity scores to DataFrame
    df['Similarity_Score'] = similarity_scores

    # Sort top 3 job recommendations
    top_recommendations = df.sort_values(by='Similarity_Score', ascending=False).head(3)

    return top_recommendations[['Job_Role', 'Similarity_Score']]

# Test with 3 user inputs (as required by the PDF instructions)
test_inputs = ["Python", "Cloud", "Automation"]
results = recommend_job_roles(test_inputs)

print(f"User Input Skills: {test_inputs}\n")
print("--- Top 3 Recommended Roles ---")
for idx, row in results.iterrows():
    print(f"Role: {row['Job_Role']} | Match Score: {row['Similarity_Score'] * 100:.2f}%")

User Input Skills: ['Python', 'Cloud', 'Automation']

--- Top 3 Recommended Roles ---
Role: DevOps Engineer | Match Score: 46.18%
Role: Cloud Architect | Match Score: 37.91%
Role: System Administrator | Match Score: 16.18%


Step 3: Create Interactive Input Interface & Cold Start Handling

In [3]:
def interactive_recommendation():
    print("=== DecodeLabs AI Job Recommendation Engine ===")
    print("Please enter at least 3 skills separated by commas (e.g., Python, Cloud, Automation):")

    user_raw_input = input("Enter skills: ")
    skills_list = [s.strip() for s in user_raw_input.split(',') if s.strip()]

    # Check for Cold Start / Insufficient Input
    if len(skills_list) < 3:
        print("\n[Warning] Cold Start Detected! Minimum 3 skills are required for accurate pattern matching.")
        print("Fallback Option: Displaying Trending / Default Career Roles:")
        df_default = pd.read_csv('skills_dataset.csv')
        for role in df_default['Job_Role'].head(3):
            print(f"- {role} (Popular Choice)")
        return

    # Run TF-IDF Recommendation Engine
    recommendations = recommend_job_roles(skills_list)

    print(f"\nProcessing recommendation logic for: {skills_list}\n")
    print("--- Top Recommended Career Paths ---")
    for idx, row in recommendations.iterrows():
        print(f"-> {row['Job_Role']} | Match Score: {row['Similarity_Score'] * 100:.2f}%")

# Execute interactive session
interactive_recommendation()

=== DecodeLabs AI Job Recommendation Engine ===
Please enter at least 3 skills separated by commas (e.g., Python, Cloud, Automation):
Enter skills: Python, Machine Learning, Deep Learning

Processing recommendation logic for: ['Python', 'Machine Learning', 'Deep Learning']

--- Top Recommended Career Paths ---
-> AI/ML Engineer | Match Score: 74.22%
-> Data Scientist | Match Score: 39.98%
-> Cybersecurity Specialist | Match Score: 6.26%
